# King County House Price Analysis

## Project overview
This project explores residential property sales in King County, Washington, and builds baseline regression models to estimate sale prices. It demonstrates data cleaning, exploratory analysis, model evaluation, and business interpretation.

**Dataset:** IBM Skills Network / Coursera King County housing dataset (`kc_house_data_NaN.csv`). This is historical educational data, not a live valuation source.

## Business questions
1. Which property characteristics are associated with higher sale prices?
2. How strongly is living area related to sale price?
3. Do location, grade, and waterfront access help explain price variation?
4. How do regression models perform on unseen data?

## 1. Setup
These packages are normally available in Google Colab. No package upgrade is needed.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2. Load and inspect the data

In [ ]:
DATA_URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/FinalModule_Coursera/data/kc_house_data_NaN.csv'
df = pd.read_csv(DATA_URL)
print('Rows and columns:', df.shape)
display(df.head())
display(df.describe().T)
print('Duplicate rows:', df.duplicated().sum())
display(df.isna().sum().sort_values(ascending=False).head(15))

## 3. Clean the data
Remove identifier columns and impute missing bedroom and bathroom values with their medians. Imputation parameters are later fitted inside model pipelines using training data only.

In [ ]:
df = df.drop(columns=['Unnamed: 0', 'id'], errors='ignore').copy()
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['sale_year'] = df['date'].dt.year
for column in ['bedrooms', 'bathrooms']:
    if column in df.columns:
        df[column] = df[column].fillna(df[column].median())
print('Remaining missing values:', int(df.isna().sum().sum()))
display(df[['price', 'bedrooms', 'bathrooms']].describe())

## 4. Exploratory data analysis
Review the target distribution and relationships between price, living area, and waterfront access.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df['price'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribution of sale prices')
axes[0].set_xlabel('Sale price ($)')
sns.boxplot(x=df['price'], ax=axes[1])
axes[1].set_title('Sale price outliers')
plt.tight_layout()
plt.show()
print('Mean sale price: ${:,.0f}'.format(df['price'].mean()))
print('Median sale price: ${:,.0f}'.format(df['price'].median()))

In [ ]:
sample = df.sample(min(3000, len(df)), random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=sample, x='sqft_living', y='price', alpha=0.35, ax=axes[0])
axes[0].set_title('Living area vs. sale price')
sns.boxplot(data=df, x='waterfront', y='price', ax=axes[1])
axes[1].set_title('Price by waterfront access')
axes[1].set_ylim(0, df['price'].quantile(0.99))
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(12, 8))
sns.heatmap(corr[['price']].sort_values('price', ascending=False), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Numeric feature correlations with sale price')
plt.tight_layout()
plt.show()

## 5. Prepare features and split the data
A held-out test set provides a more realistic estimate of performance on unseen properties. Preprocessing is kept inside scikit-learn pipelines to prevent fitting transformations on test data.

In [ ]:
candidate_features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'lat', 'long', 'sqft_living15', 'sqft_lot15']
features = [column for column in candidate_features if column in df.columns]
X = df[features].copy()
y = df['price'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
print('Features:', features)
print('Training rows:', len(X_train), '| Test rows:', len(X_test))

## 6. Train and compare models
Compare a mean-price baseline with linear regression and Ridge regression. Lower MAE and RMSE indicate smaller prediction errors; higher R-squared indicates more variance explained.

In [ ]:
baseline_pred = np.repeat(y_train.mean(), len(y_test))
linear_model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', LinearRegression())])
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)
ridge_model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))])
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
def metrics(y_true, y_pred, name):
    return {'Model': name, 'MAE ($)': mean_absolute_error(y_true, y_pred), 'RMSE ($)': np.sqrt(mean_squared_error(y_true, y_pred)), 'R-squared': r2_score(y_true, y_pred)}
results = pd.DataFrame([metrics(y_test, baseline_pred, 'Mean baseline'), metrics(y_test, linear_pred, 'Linear regression'), metrics(y_test, ridge_pred, 'Ridge regression')]).sort_values('RMSE ($)')
display(results)

## 7. Cross-validation
Cross-validation is performed on training data only. The held-out test set remains reserved for final evaluation.

In [ ]:
cv_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))])
cv_scores = cross_val_score(cv_pipeline, X_train, y_train, cv=5, scoring='r2')
print('Five-fold R-squared scores:', np.round(cv_scores, 3))
print('Mean CV R-squared: {:.3f}'.format(cv_scores.mean()))
print('CV standard deviation: {:.3f}'.format(cv_scores.std()))

## 8. Prediction diagnostics

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=ridge_pred, alpha=0.35)
low = min(y_test.min(), ridge_pred.min())
high = max(y_test.max(), ridge_pred.max())
plt.plot([low, high], [low, high], 'r--', label='Perfect prediction')
plt.xlabel('Actual sale price ($)')
plt.ylabel('Predicted sale price ($)')
plt.title('Ridge regression: actual vs. predicted')
plt.legend()
plt.tight_layout()
plt.show()
residuals = y_test - ridge_pred
sns.histplot(residuals, bins=40, kde=True)
plt.axvline(0, color='red', linestyle='--')
plt.title('Ridge model residual distribution')
plt.xlabel('Actual minus predicted price ($)')
plt.show()
display(pd.DataFrame({'Actual price': y_test, 'Predicted price': ridge_pred}).head(10))

## 9. Findings and interpretation
After running the notebook, use the generated charts and metrics to complete these observations. Do not publish performance numbers until execution is verified.

- Describe the relationship between living area and sale price.
- Compare waterfront and non-waterfront sale price distributions.
- Identify the features most strongly correlated with price.
- Compare test-set MAE, RMSE, and R-squared across models.
- Inspect residuals for systematic over- or under-prediction.

## 10. Limitations and next steps
- The dataset is historical and does not represent current market conditions.
- The data covers one county; results may not generalize to other markets.
- Renovations, school boundaries, interest rates, and detailed neighborhood characteristics are not fully represented.
- Extreme sale prices can influence regression metrics.
- This is observational data; correlation does not establish causation.

Potential improvements include tuning Ridge regularization, testing tree-based models, evaluating performance by price segment, engineering reliable property-age features, and using time-based validation.

## Conclusion
This project demonstrates a reproducible workflow for cleaning housing data, exploring price patterns, training regression models, and evaluating predictions on held-out data. Model selection should be based on test-set performance—not training score alone.

**Author:** Jerry Jerome